# Sex-Specific Significance Testing: Friedman, Wilcoxon, and DeLong

The three independent statistical tests used to establish and confirm the sex-based performance gap (Methodology, "Interpretability and Sex-Specific Performance Analysis"): a three-condition Friedman test (pooled vs. male-only vs. female-only), a paired Wilcoxon signed-rank test (male vs. female AUC), and a DeLong test applied at the individual repeated-CV fold level and combined per model-horizon block via Stouffer's method.

In [1]:
import pandas as pd
import numpy as np
import ast
from scipy.stats import wilcoxon, friedmanchisquare, norm
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, BaggingClassifier, ExtraTreesClassifier,
                               AdaBoostClassifier, StackingClassifier)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
BASE = './'
HORIZONS = ['1y', '2y', '3y', '4y', '5y']

df = pd.read_csv(BASE + 'modeling_dataset.csv')
pooled = pd.read_csv(BASE + 'repeated_cv_summary.csv')
sex_summary = pd.read_csv(BASE + 'sex_specific_repeated_cv_summary.csv')
sex_raw = pd.read_csv(BASE + 'sex_specific_repeated_cv_raw_per_fold.csv')
sex_results = pd.read_csv(BASE + 'sex_specific_results.csv')

MODELS = sex_summary['model'].unique().tolist()
print(f'{len(MODELS)} models x {len(HORIZONS)} horizons = {len(MODELS)*len(HORIZONS)} model/horizon blocks')

14 models x 5 horizons = 70 model/horizon blocks


## 1. Wilcoxon signed-rank test (male vs. female AUC, 70 model-horizon pairs)

In [2]:
# --- Option A: 70 pairs, one per (model, horizon), using mean AUC ---
pivotA = sex_summary.pivot_table(index=['model', 'horizon'], columns='sex', values='auc_mean')
diffsA = pivotA['F'] - pivotA['M']

statA, pA = wilcoxon(diffsA)
nA = len(diffsA)
rA = 1 - (2 * statA) / (nA * (nA + 1) / 2)  # matched-pairs rank-biserial correlation (effect size)

print('=== Wilcoxon, Option A (70 model/horizon pairs, mean AUC) ===')
print(f'N pairs = {nA}, W = {statA:.1f}, p = {pA:.2e}')
print(f'Median (F - M) AUC = {diffsA.median():+.4f}, all {nA} pairs favour F: {(diffsA > 0).sum()}/{nA}')
print(f'Effect size (rank-biserial r) = {rA:.4f}')

=== Wilcoxon, Option A (70 model/horizon pairs, mean AUC) ===
N pairs = 70, W = 0.0, p = 3.55e-13
Median (F - M) AUC = +0.0895, all 70 pairs favour F: 70/70
Effect size (rank-biserial r) = 1.0000


## 2. Friedman test, 3-condition (Pooled vs. Male vs. Female)

In [3]:
three_way = pooled.set_index(['model', 'horizon'])[['auc_mean']].rename(columns={'auc_mean': 'Pooled'})
three_way = three_way.join(pivotA[['M', 'F']]).dropna()
three_way.columns = ['Pooled', 'Male', 'Female']

stat3, p3 = friedmanchisquare(three_way['Pooled'], three_way['Male'], three_way['Female'])
print('=== Friedman, 3-condition (Pooled vs Male vs Female), N blocks =', len(three_way), '===')
print(f'chi-square = {stat3:.4f}, p = {p3:.2e}')
print()
print('Mean AUC by condition:')
print(three_way.mean().round(4))
print()
print('Mean rank by condition (1 = best):')
ranks = three_way.rank(axis=1, ascending=False)
print(ranks.mean().round(3))

=== Friedman, 3-condition (Pooled vs Male vs Female), N blocks = 70 ===
chi-square = 138.0286, p = 1.07e-30

Mean AUC by condition:
Pooled    0.6889
Male      0.6426
Female    0.7310
dtype: float64

Mean rank by condition (1 = best):
Pooled    1.986
Male      3.000
Female    1.014
dtype: float64


## 3. DeLong test, applied at the repeated-CV fold level, combined via Stouffer's method

In [4]:
MODEL_CONFIGS = {
    'Logistic Regression': dict(cls=LogisticRegression, scaled=True,
        fixed_kwargs=dict(solver='saga', max_iter=5000, random_state=RANDOM_STATE)),
    'Random Forest': dict(cls=RandomForestClassifier, scaled=False,
        fixed_kwargs=dict(random_state=RANDOM_STATE, n_jobs=-1)),
    'XGBoost': dict(cls=XGBClassifier, scaled=False,
        fixed_kwargs=dict(random_state=RANDOM_STATE, eval_metric='logloss'), needs_scale_pos_weight=True),
    'LightGBM': dict(cls=LGBMClassifier, scaled=False,
        fixed_kwargs=dict(random_state=RANDOM_STATE, class_weight='balanced', verbose=-1)),
    'SVM': dict(cls=SVC, scaled=True, fixed_kwargs=dict(probability=True, random_state=RANDOM_STATE)),
    'Decision Tree': dict(cls=DecisionTreeClassifier, scaled=False, fixed_kwargs=dict(random_state=RANDOM_STATE)),
    'Naive Bayes': dict(cls=GaussianNB, scaled=True, fixed_kwargs=dict()),
    'KNN': dict(cls=KNeighborsClassifier, scaled=True, fixed_kwargs=dict()),
    'Bagging (DT base)': dict(cls=BaggingClassifier, scaled=False,
        fixed_kwargs=dict(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE, n_jobs=-1)),
    'Extra Trees': dict(cls=ExtraTreesClassifier, scaled=False, fixed_kwargs=dict(random_state=RANDOM_STATE, n_jobs=-1)),
    'Bagging (KNN base)': dict(cls=BaggingClassifier, scaled=True,
        fixed_kwargs=dict(estimator=KNeighborsClassifier(), random_state=RANDOM_STATE, n_jobs=-1)),
    'AdaBoost': dict(cls=AdaBoostClassifier, scaled=False, fixed_kwargs=dict(random_state=RANDOM_STATE)),
    'Stacking (diverse)': dict(cls=StackingClassifier, scaled=True,
        fixed_kwargs=dict(estimators=[
            ('lr', LogisticRegression(solver='saga', max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE)),
            ('rf', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
            ('svm', SVC(probability=True, class_weight='balanced', random_state=RANDOM_STATE))],
            final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), cv=5, n_jobs=-1)),
    'Stacking (boosting)': dict(cls=StackingClassifier, scaled=True,
        fixed_kwargs=dict(estimators=[
            ('xgb', XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')),
            ('lgbm', LGBMClassifier(class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)),
            ('ada', AdaBoostClassifier(random_state=RANDOM_STATE))],
            final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), cv=5, n_jobs=-1)),
}

COMORBID_COLS = [c for c in df.columns if c not in (
    ['person_id', 'sex', 'age', 'postcode', 'rurality', 'ses_irsd_decile', 'ses_missing',
     'incident_cvd', 'years_followup', 'split',
     'baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
     'baseline_n_episodes', 'baseline_history_days']
    + [f'label_{h}y' for h in range(1, 6)] + [f'eligible_{h}y' for h in range(1, 6)]
)]
NUMERIC_COLS = ['age', 'baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
                'baseline_n_episodes', 'baseline_history_days']
FEATURE_COLS = NUMERIC_COLS + COMORBID_COLS

def make_preprocessor(scaled):
    num_step = StandardScaler() if scaled else 'passthrough'
    return ColumnTransformer([('num', num_step, NUMERIC_COLS), ('bin', 'passthrough', COMORBID_COLS)])

def delong_auc_var(y_true, y_score):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score)
    pos = y_score[y_true == 1]; neg = y_score[y_true == 0]
    m, n = len(pos), len(neg)
    diff = pos.reshape(-1, 1) - neg.reshape(1, -1)
    psi = (diff > 0).astype(float) + 0.5 * (diff == 0).astype(float)
    V10 = psi.mean(axis=1); V01 = psi.mean(axis=0)
    auc = psi.mean()
    var_auc = V10.var(ddof=1) / m + V01.var(ddof=1) / n
    return auc, var_auc, m, n

print(f'{len(FEATURE_COLS)} features, {len(MODEL_CONFIGS)} models configured')

18 features, 14 models configured


In [5]:
from sklearn.model_selection import StratifiedKFold

RANDOM_STATE_CV = 42
SEEDS = list(range(10))
delong_cv_rows = []
delong_cv_failures = []

for model_name, cfg in MODEL_CONFIGS.items():
    for h in HORIZONS:
        elig_col, label_col = f'eligible_{h}', f'label_{h}'
        for sex in ['M', 'F']:
            row = sex_results[(sex_results.model == model_name) & (sex_results.sex == sex) & (sex_results.horizon == h)]
            if len(row) == 0:
                continue
            best_params = {k.replace('clf__', ''): v for k, v in ast.literal_eval(row.iloc[0]['best_params']).items()}

            sub = df[(df[elig_col]) & (df['sex'] == sex)]
            X_all, y_all = sub[FEATURE_COLS], sub[label_col].astype(int)
            if len(X_all) < 30 or y_all.nunique() < 2:
                continue

            preprocessor = make_preprocessor(cfg['scaled'])
            fold_idx = 0
            for seed in SEEDS:
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
                for train_idx, test_idx in skf.split(X_all, y_all):
                    X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
                    y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]
                    if y_train.nunique() < 2 or y_test.nunique() < 2:
                        fold_idx += 1
                        continue

                    init_kwargs = dict(cfg['fixed_kwargs'])
                    if cfg.get('needs_scale_pos_weight'):
                        init_kwargs['scale_pos_weight'] = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

                    pipe = Pipeline([('preprocess', preprocessor), ('clf', cfg['cls'](**init_kwargs))])
                    pipe.set_params(**{f'clf__{k}': v for k, v in best_params.items()})
                    try:
                        pipe.fit(X_train, y_train)
                        proba = pipe.predict_proba(X_test)[:, 1]
                        if not np.all(np.isfinite(proba)):
                            raise ValueError('non-finite probabilities')
                        auc, var_auc, m, n = delong_auc_var(y_test.values, proba)
                        delong_cv_rows.append(dict(model=model_name, horizon=h, sex=sex, seed=seed, fold=fold_idx,
                                                    auc=auc, var_auc=var_auc, m=m, n=n))
                    except Exception as e:
                        delong_cv_failures.append(dict(model=model_name, horizon=h, sex=sex, seed=seed, fold=fold_idx, error=str(e)))
                    fold_idx += 1

delong_cv_df = pd.DataFrame(delong_cv_rows)
delong_cv_fail_df = pd.DataFrame(delong_cv_failures)
print(f'{len(delong_cv_df)} fold-level DeLong fits completed, {len(delong_cv_fail_df)} failed/skipped')
delong_cv_df.to_csv(BASE + 'significance_testB_delong_repeatedcv_raw.csv', index=False)
if len(delong_cv_fail_df) > 0:
    delong_cv_fail_df.to_csv(BASE + 'significance_testB_delong_repeatedcv_failures.csv', index=False)

7000 fold-level DeLong fits completed, 0 failed/skipped


In [6]:
# Pair M and F within each (model, horizon, seed, fold) block and compute per-fold DeLong z/p
piv = delong_cv_df.pivot_table(index=['model', 'horizon', 'seed', 'fold'],
                                columns='sex', values=['auc', 'var_auc'])
piv = piv.dropna()

z_scores = (piv[('auc', 'F')] - piv[('auc', 'M')]) / np.sqrt(piv[('var_auc', 'F')] + piv[('var_auc', 'M')])
p_values = 2 * (1 - norm.cdf(z_scores.abs()))

per_fold = pd.DataFrame({'diff': piv[('auc', 'F')] - piv[('auc', 'M')], 'z': z_scores, 'p': p_values}).reset_index()

summary_rows = []
for (model_name, h), grp in per_fold.groupby(['model', 'horizon']):
    summary_rows.append(dict(
        model=model_name, horizon=h, n_folds=len(grp),
        pct_favour_F=(grp['diff'] > 0).mean() * 100,
        pct_individually_significant=(grp['p'] < 0.05).mean() * 100,
        mean_z=grp['z'].mean(), median_diff=grp['diff'].median(),
        stouffer_z=grp['z'].sum() / np.sqrt(len(grp)),
    ))

delong_cv_summary = pd.DataFrame(summary_rows)
delong_cv_summary['stouffer_p'] = 2 * (1 - norm.cdf(delong_cv_summary['stouffer_z'].abs()))

print('=== Per-fold DeLong (repeated CV), summarised per model/horizon ===')
print(f'Total model/horizon blocks: {len(delong_cv_summary)}')
print(f'Blocks where >=50% of folds favour F: {(delong_cv_summary.pct_favour_F >= 50).sum()}/{len(delong_cv_summary)}')
print(f'Blocks with Stouffer-combined p<0.05: {(delong_cv_summary.stouffer_p < 0.05).sum()}/{len(delong_cv_summary)}')
print()
print(delong_cv_summary.groupby('horizon')[['pct_favour_F', 'pct_individually_significant', 'stouffer_z', 'stouffer_p']].mean().round(4))
delong_cv_summary.to_csv(BASE + 'significance_testB_delong_repeatedcv_summary.csv', index=False)
per_fold.to_csv(BASE + 'significance_testB_delong_repeatedcv_perfold.csv', index=False)

=== Per-fold DeLong (repeated CV), summarised per model/horizon ===
Total model/horizon blocks: 70
Blocks where >=50% of folds favour F: 70/70
Blocks with Stouffer-combined p<0.05: 70/70

         pct_favour_F  pct_individually_significant  stouffer_z  stouffer_p
horizon                                                                    
1y            92.8571                       26.1429      9.7543      0.0000
2y            96.4286                       37.0000     11.5934      0.0000
3y            92.2857                       25.2857      9.4424      0.0000
4y            91.8571                       16.0000      7.8183      0.0000
5y            81.8571                       11.2857      5.9433      0.0001


### Per-horizon combined z-statistics

DeLong figures (z = 5.94-11.59, all p < 0.0002) average the 14 per-model-horizon Stouffer-combined z-statistics computed above, within each horizon.


In [7]:
horizon_summary = delong_cv_summary.groupby('horizon')[['pct_favour_F', 'pct_individually_significant', 'stouffer_z', 'stouffer_p']].mean().round(4)
print(horizon_summary)

         pct_favour_F  pct_individually_significant  stouffer_z  stouffer_p
horizon                                                                    
1y            92.8571                       26.1429      9.7543      0.0000
2y            96.4286                       37.0000     11.5934      0.0000
3y            92.2857                       25.2857      9.4424      0.0000
4y            91.8571                       16.0000      7.8183      0.0000
5y            81.8571                       11.2857      5.9433      0.0001
